# 🚀 AIC 2026 — End-to-End Pipeline Tutorial

Notebook này hướng dẫn và tự động thực thi toàn bộ pipeline của hệ thống Video Search Agent: từ việc chuẩn bị, giải nén dữ liệu BTC, tạo metadata, cho đến việc huấn luyện model (Projection Head, Temporal) và đẩy vectors lên Vector Database.

## Cài đặt thư viện cần thiết
Trước khi bắt đầu, hãy đảm bảo bạn đã cài đặt đủ thư viện.

In [ ]:
!pip install -r requirements.txt

## Bước 1: Giải nén dữ liệu từ Ban Tổ Chức (BTC)
Code sẽ tự động lấy các file `.zip` nằm trong thư mục `ZIP/` và giải nén chúng vào đúng cấu trúc thư mục con trong `data/` để model có thể đọc được.

In [ ]:
# Xem trước các file sẽ được giải nén (Dry run)
!python scripts/extract_btc_data.py --dry-run

In [ ]:
# Tiến hành giải nén thực tế
!python scripts/extract_btc_data.py

## Bước 2: Import Metadata
Bước này quét toàn bộ file JSON, CSV, ảnh và map keyframes từ BTC đã giải nén để tổng hợp thành một file duy nhất `data/index/metadata.jsonl` chứa đầy đủ thông tin (frame_id, pts_time, fps, object tags...).

In [ ]:
!python scripts/import_btc_data.py

## Bước 3: Huấn luyện Projection Head
Fine-tune Projection Head để đưa text features và image features (đã chiết xuất từ CLIP) về cùng một không gian vector bằng InfoNCE Loss. 
Kết quả sẽ được lưu tại: `data/index/projection_head.pt`.

In [ ]:
!python -m backend.training.train

## Bước 4: Huấn luyện Temporal Video Encoder
Sử dụng GRU (Gated Recurrent Unit) layer để học sự phụ thuộc giữa các keyframes theo thời gian, giúp tổng hợp feature của cả đoạn video.
Kết quả sẽ được lưu tại: `data/index/temporal_encoder.pt`.

In [ ]:
!python -m backend.training.train_temporal

## Bước 5: Push Vectors lên Vector Database (Qdrant)
Đẩy các feature vectors của toàn bộ khung hình lên Qdrant Database. 
Nếu sử dụng môi trường Local/FAISS, bạn có thể chuyển qua bước 6.

In [ ]:
# Đẩy vectors lên database (Sử dụng Qdrant)
# Mặc định push raw CLIP 512d để đạt hiệu quả Zero-shot tốt nhất.
!python backend/embedding/push_to_remote.py --recreate

## Bước 6: Build FAISS Index Local (Tùy chọn thay thế)
Sử dụng nếu bạn không muốn chạy Qdrant mà chỉ muốn search local bằng FAISS.

In [ ]:
!python scripts/build_index_features.py